# 00 - Setup and environment

Run once per Colab session. Creates the Drive tree, installs dependencies, verifies the runtime.

**Note on storage:** Pro+ gives 2 TB of *Drive*, not RAM. High-RAM runtimes cap near 51 GB (~83 GB on A100), so all training data is staged to `/content` and read from local SSD.

In [ ]:
# --- standard header: every notebook starts with exactly this ---
from google.colab import drive; drive.mount('/content/drive')

REPO = '/content/secure-dns-trust-ai'
!git -C {REPO} pull -q 2>/dev/null || git clone -q https://github.com/sandesh20lamichhane/secure-dns-trust-ai.git {REPO}

import sys, os; sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'
%load_ext autoreload
%autoreload 2

from src.utils import config, manifest, seeds, io
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install dnspython cryptography xgboost shap pyarrow zstandard pyyaml

In [ ]:
import torch, psutil, shutil
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('RAM GB:', round(psutil.virtual_memory().total/1e9, 1))
print('local disk free GB:', round(shutil.disk_usage('/content').free/1e9, 1))

In [ ]:
import json
for section in ('data','artifacts','results'):
    for k, v in P[section].items(): print(f'{section}.{k:14s} {v}')
print('manifest:', P['manifest'])

In [ ]:
# Seed the leakage notes file on first run
from pathlib import Path
notes = Path(P['leakage_notes'])
if not notes.exists():
    notes.write_text('# Leakage audit log\n\nOne entry per suspicious feature: '
                     'feature / why suspicious / investigation / decision / reason.\n')
print(notes.read_text()[:300])